In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# ================== Setup Directory Structure ==================
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Artifact directories
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
AUDIT_DIR = ARTIFACTS_DIR / "audit"
MODELS_DIR = ARTIFACTS_DIR / "models"
PLOTS_DIR = ARTIFACTS_DIR / "plots"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
TABLES_DIR = ARTIFACTS_DIR / "tables"

# Raw file
RAW_DATA_PATH = RAW_DIR / "CompaniesHouseData-2026-03-02.csv"

# Create directories
for folder in [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    ARTIFACTS_DIR,
    AUDIT_DIR,
    TABLES_DIR,
    PLOTS_DIR,
    REPORTS_DIR,
    MODELS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)

Project root: /Users/lingzitong/Desktop
Raw data path: /Users/lingzitong/Desktop/data/raw/CompaniesHouseData-2026-03-02.csv


In [3]:
# Upload the original data set
raw_data = pd.read_csv(RAW_DATA_PATH, encoding="utf-8")
print('Raw shape:', raw_data.shape)
print('Columns:', len(raw_data.columns))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lingzitong/Desktop/data/raw/CompaniesHouseData-2026-03-02.csv'

In [ ]:
df.info()

In [ ]:
expected_columns = [
    "CompanyNumber",
    "CompanyName",
    "CompanyCategory",
    "CompanyStatus",
    "CountryofOrigin",
    "IncorporationDate",
    "AccountsCategory",
    "SICCode1",
    "SICCode2",
    "SICCode3",
    "SICCode4",
    "PostTown",
    "County",
    "Country",
    "PostCode",
    "URI",
]

missing_expected = [col for col in expected_columns if col not in df.columns]
present_expected = [col for col in expected_columns if col in df.columns]

print("Present expected columns:", present_expected)
print("Missing expected columns:", missing_expected)

In [ ]:
missing_summary = (
    df[present_expected]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_ratio")
    .to_frame()
)

missing_summary

In [ ]:
for col in ["CompanyStatus", "CompanyCategory", "CountryofOrigin", "AccountsCategory"]:
    if col in df.columns:
        print(f"\n===== {col} =====")
        print(df[col].value_counts(dropna=False).head(20))

In [ ]:
selected_columns = [col for col in expected_columns if col in df.columns]

merchant_master = df[selected_columns].copy()

print("Selected shape:", merchant_master.shape)
merchant_master.head()

In [ ]:
rename_map = {
    "CompanyNumber": "merchant_id",
    "CompanyName": "merchant_name",
    "CompanyCategory": "company_category",
    "CompanyStatus": "company_status",
    "CountryofOrigin": "country_of_origin",
    "IncorporationDate": "incorporation_date",
    "AccountsCategory": "accounts_category",
    "SICCode1": "sic_code_1",
    "SICCode2": "sic_code_2",
    "SICCode3": "sic_code_3",
    "SICCode4": "sic_code_4",
    "PostTown": "post_town",
    "County": "county",
    "Country": "country",
    "PostCode": "postcode",
    "URI": "source_uri",
}

merchant_master = merchant_master.rename(columns=rename_map)
merchant_master.head()

In [ ]:
print("Unique merchant_id count:", merchant_master["merchant_id"].nunique())
print("Total rows:", len(merchant_master))

duplicate_ids = merchant_master["merchant_id"].duplicated().sum()
print("Duplicated merchant_id rows:", duplicate_ids)

In [ ]:
today = pd.Timestamp.today().normalize()

merchant_master["company_age_years"] = (
    (today - merchant_master["incorporation_date"]).dt.days / 365.25
).round(2)

merchant_master[["merchant_id", "merchant_name", "incorporation_date", "company_age_years"]].head()

In [ ]:
merchant_master["source_system"] = "Companies House"
merchant_master["snapshot_date"] = pd.Timestamp("2026-03-02")

In [ ]:
summary = {
    "rows": len(merchant_master),
    "columns": merchant_master.shape[1],
    "unique_merchants": merchant_master["merchant_id"].nunique(),
    "missing_merchant_name_ratio": merchant_master["merchant_name"].isna().mean(),
    "missing_company_status_ratio": merchant_master["company_status"].isna().mean(),
    "missing_incorporation_date_ratio": merchant_master["incorporation_date"].isna().mean(),
}

summary

In [ ]:
output_file = PROCESSED_DIR / "merchant_master_selected.csv"
merchant_master.to_csv(output_file, index=False)

print("Saved to:", output_file)